Window Components



In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("MySparkSessionDemo")\
    .master("local[*]")\
    .config("spark.sql.shuffle.partitions","4")\
    .getOrCreate()

print("SparkSession created")
print("AppName",spark.sparkContext.appName)
print("master",spark.sparkContext.master)
print("Spark Version",spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 21:44:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession created
AppName MySparkSessionDemo
master local[*]
Spark Version 3.5.4


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("WindowFunctions").getOrCreate()

In [2]:
data = [
    (1, "Alice", "HR", "Mumbai", 50000),
    (2, "Bob", "IT", "Bangalore", 70000),
    (3, "Charlie", "IT", "Mumbai", 80000),
    (4, "David", "HR", "Delhi", 60000),
    (5, "Eve", "Sales", "Mumbai", 75000),
    (6, "Frank", "Sales", "Delhi", 72000),
    (7, "Grace", "IT", "Bangalore", 90000),
    (8, "Helen", "HR", "Mumbai", 65000)
]

columns = ["id", "name", "department", "city", "salary"]

df = spark.createDataFrame(data, columns)
df.show()

+---+-------+----------+---------+------+
| id|   name|department|     city|salary|
+---+-------+----------+---------+------+
|  1|  Alice|        HR|   Mumbai| 50000|
|  2|    Bob|        IT|Bangalore| 70000|
|  3|Charlie|        IT|   Mumbai| 80000|
|  4|  David|        HR|    Delhi| 60000|
|  5|    Eve|     Sales|   Mumbai| 75000|
|  6|  Frank|     Sales|    Delhi| 72000|
|  7|  Grace|        IT|Bangalore| 90000|
|  8|  Helen|        HR|   Mumbai| 65000|
+---+-------+----------+---------+------+



# Window functions perform calculations across a set of rows
# WITHOUT reducing rows (unlike groupBy)

# Syntax:
# Window.partitionBy().orderBy()

# General Syntax:

# Window.partitionBy("col").orderBy("col").rowsBetween(start, end)

# Components:
# 1. partitionBy → grouping
# 2. orderBy → sorting inside group
# 3. rowsBetween → frame (range of rows)

In [ ]:
window_spec = Window.partitionBy("department")

df.withColumn(
    "avg_salary",
    F.avg("salary").over(window_spec)
).show()

In [ ]:
window_spec = Window.partitionBy("department").orderBy(F.col("salary").desc())

df.withColumn(
    "ordered_salary",
    F.row_number().over(window_spec)
).show()

RANKING FUNCTIONS

In [ ]:
window_spec = Window.partitionBy("department").orderBy(F.col("salary").desc())

df.withColumn(
    "row_number",
    F.row_number().over(window_spec)
).show()

In [ ]:
df.withColumn(
    "rank",
    F.rank().over(window_spec)
).show()

In [ ]:
df.withColumn(
    "dense_rank",
    F.dense_rank().over(window_spec)
).show()

In [ ]:
from pyspark.sql.window import Window

data = [
    (101, "Alice",   "HR",    "Mumbai",    "2024-01-10", 50000),
    (102, "Bob",     "IT",    "Bangalore", "2024-01-12", 70000),
    (103, "Charlie", "IT",    "Mumbai",    "2024-01-15", 80000),
    (104, "David",   "HR",    "Delhi",     "2024-01-18", 60000),
    (105, "Eve",     "Sales", "Mumbai",    "2024-01-20", 75000),
    (106, "Frank",   "Sales", "Delhi",     "2024-01-22", 72000),
    (107, "Grace",   "IT",    "Bangalore", "2024-01-25", 90000),
    (108, "Helen",   "HR",    "Mumbai",    "2024-01-28", 65000),
    (109, "Ivy",     "IT",    "Delhi",     "2024-02-01", 70000),
    (110, "Jack",    "Sales", "Bangalore", "2024-02-03", 68000),
    (111, "Kevin",   "HR",    "Delhi",     "2024-02-05", 60000),
    (112, "Lara",    "IT",    "Mumbai",    "2024-02-08", 85000),
    (113, "Mona",    "Sales", "Mumbai",    "2024-02-10", 75000),
    (114, "Nina",    "HR",    "Bangalore", "2024-02-12", 58000),
    (115, "Oscar",   "IT",    "Delhi",     "2024-02-15", 95000),
    (116, "Paul",    "Sales", "Delhi",     "2024-02-18", 72000),
    (117, "Quinn",   "HR",    "Mumbai",    "2024-02-20", 67000),
    (118, "Rose",    "IT",    "Bangalore", "2024-02-22", 88000),
    (119, "Sam",     "Sales", "Mumbai",    "2024-02-25", 79000),
    (120, "Tina",    "HR",    "Delhi",     "2024-02-28", 61000)
]

columns = ["emp_id", "name", "department", "city", "joining_date", "salary"]

df = spark.createDataFrame(data, columns)

df = df.withColumn("joining_date", F.to_date("joining_date", "yyyy-MM-dd"))

df.show(truncate=False)
df.printSchema()

LEVEL 1 — BASICS (Foundation)


1. Assign row number to employees within each department based on salary (highest first)
2. Rank employees within each department based on salary
3. Find dense rank of employees within each department
4. Get top 3 highest paid employees per department
5. Get bottom 2 lowest paid employees per department
6. Add department average salary to each employee
7. Add department total salary to each employee
8. Find total employee count per department using window

LEVEL 2 — INTERMEDIATE (Real Logic)


9. Find employees earning more than their department average
10. Find employees earning less than their department average
11. Find highest salary employee in each department
12. Find second highest salary per department
13. Find salary difference between current and previous employee (within department ordered by salary)
14. Add previous salary column using lag (department-wise)
15. Add next salary column using lead (department-wise)
16. Find salary increase between consecutive employees (within department)
17. Calculate running total salary per department based on joining_date
18. Calculate cumulative average salary per department based on joining_date


LEVEL 3 — ADVANCED (REAL-TIME)


19. Find top 2 cities with highest total salary per department
20. Calculate department contribution percentage using window function
21. Find employee whose salary is closest to department average
22. Find employees whose salary is higher than previous employee
23. Find first and last salary per department based on joining_date
24. Deduplicate employees: keep latest record per employee (based on joining_date)

In [3]:
df.show()

+---+-------+----------+---------+------+
| id|   name|department|     city|salary|
+---+-------+----------+---------+------+
|  1|  Alice|        HR|   Mumbai| 50000|
|  2|    Bob|        IT|Bangalore| 70000|
|  3|Charlie|        IT|   Mumbai| 80000|
|  4|  David|        HR|    Delhi| 60000|
|  5|    Eve|     Sales|   Mumbai| 75000|
|  6|  Frank|     Sales|    Delhi| 72000|
|  7|  Grace|        IT|Bangalore| 90000|
|  8|  Helen|        HR|   Mumbai| 65000|
+---+-------+----------+---------+------+



In [5]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec=Window.partitionBy("department")
df.withColumn("avg_salary",F.avg("salary").over(window_spec)).show()

+---+-------+----------+---------+------+------------------+
| id|   name|department|     city|salary|        avg_salary|
+---+-------+----------+---------+------+------------------+
|  1|  Alice|        HR|   Mumbai| 50000|58333.333333333336|
|  4|  David|        HR|    Delhi| 60000|58333.333333333336|
|  8|  Helen|        HR|   Mumbai| 65000|58333.333333333336|
|  2|    Bob|        IT|Bangalore| 70000|           80000.0|
|  3|Charlie|        IT|   Mumbai| 80000|           80000.0|
|  7|  Grace|        IT|Bangalore| 90000|           80000.0|
|  5|    Eve|     Sales|   Mumbai| 75000|           73500.0|
|  6|  Frank|     Sales|    Delhi| 72000|           73500.0|
+---+-------+----------+---------+------+------------------+



from pyspark.sql.window import Window 
window_spec=Window.partitionBy("col1").orderby("col2")


window_spec=Window.partitionBy("col1").orderBy("col2").rowsBetween(start,end)

df.withColumn(
    "new_column",
    F.function_name("col").over(window_spec)
)

Ranking - row_number,rank,dense_rank # Always orderBy 

Aggegated - sum,avg,count,min,max

Analytical - lag,lead,first_value,last_value 

cumulative - running_sum, running_avg 

Ranking :

1. row_number() :

    - Assisgns unique sequential number 
    - No Duplicates 
    - Even if same value --> numbers are diffrent 



    window_spec=Window.partitionBy("department").orderBy(F.col("salary").desc())

    df.withColumn(
        "row_num",
        F.row_number().over(window_spec)
    ).show()


2. Rank()

    Same Value -- Same rank 

    But it will Skips the number (gaps)





In [6]:
window_spec=Window.partitionBy("department").orderBy(F.col("salary").desc())

df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
).show()

+---+-------+----------+---------+------+-------+
| id|   name|department|     city|salary|row_num|
+---+-------+----------+---------+------+-------+
|  8|  Helen|        HR|   Mumbai| 65000|      1|
|  4|  David|        HR|    Delhi| 60000|      2|
|  1|  Alice|        HR|   Mumbai| 50000|      3|
|  7|  Grace|        IT|Bangalore| 90000|      1|
|  3|Charlie|        IT|   Mumbai| 80000|      2|
|  2|    Bob|        IT|Bangalore| 70000|      3|
|  5|    Eve|     Sales|   Mumbai| 75000|      1|
|  6|  Frank|     Sales|    Delhi| 72000|      2|
+---+-------+----------+---------+------+-------+



In [7]:
window_spec=Window.partitionBy("department").orderBy(F.col("salary").desc())

df.withColumn(
    "rank",
    F.rank().over(window_spec)
).show()

+---+-------+----------+---------+------+----+
| id|   name|department|     city|salary|rank|
+---+-------+----------+---------+------+----+
|  8|  Helen|        HR|   Mumbai| 65000|   1|
|  4|  David|        HR|    Delhi| 60000|   2|
|  1|  Alice|        HR|   Mumbai| 50000|   3|
|  7|  Grace|        IT|Bangalore| 90000|   1|
|  3|Charlie|        IT|   Mumbai| 80000|   2|
|  2|    Bob|        IT|Bangalore| 70000|   3|
|  5|    Eve|     Sales|   Mumbai| 75000|   1|
|  6|  Frank|     Sales|    Delhi| 72000|   2|
+---+-------+----------+---------+------+----+



In [8]:
window_spec=Window.partitionBy("department").orderBy(F.col("salary").desc())

df.withColumn(
    "dense_rank",
    F.dense_rank().over(window_spec)
).show()

+---+-------+----------+---------+------+----------+
| id|   name|department|     city|salary|dense_rank|
+---+-------+----------+---------+------+----------+
|  8|  Helen|        HR|   Mumbai| 65000|         1|
|  4|  David|        HR|    Delhi| 60000|         2|
|  1|  Alice|        HR|   Mumbai| 50000|         3|
|  7|  Grace|        IT|Bangalore| 90000|         1|
|  3|Charlie|        IT|   Mumbai| 80000|         2|
|  2|    Bob|        IT|Bangalore| 70000|         3|
|  5|    Eve|     Sales|   Mumbai| 75000|         1|
|  6|  Frank|     Sales|    Delhi| 72000|         2|
+---+-------+----------+---------+------+----------+



In [9]:
data = [
    (1, "Alice",   "HR",    "Mumbai",    "2024-01-01", 50000),
    (2, "Bob",     "IT",    "Bangalore", "2024-01-02", 70000),
    (3, "Charlie", "IT",    "Mumbai",    "2024-01-03", 80000),
    (4, "David",   "HR",    "Delhi",     "2024-01-04", 60000),
    (5, "Eve",     "Sales", "Mumbai",    "2024-01-05", 75000),
    (6, "Frank",   "Sales", "Delhi",     "2024-01-06", 72000),
    (7, "Grace",   "IT",    "Bangalore", "2024-01-07", 90000),
    (8, "Helen",   "HR",    "Mumbai",    "2024-01-08", 65000),
    (9, "Ivy",     "IT",    "Delhi",     "2024-01-09", 70000),
    (10, "Jack",   "Sales", "Bangalore", "2024-01-10", 68000),
    (11, "Kevin",  "HR",    "Delhi",     "2024-01-11", 60000),
    (12, "Lara",   "IT",    "Mumbai",    "2024-01-12", 85000),
    (13, "Mona",   "Sales", "Mumbai",    "2024-01-13", 75000),
    (14, "Nina",   "HR",    "Bangalore", "2024-01-14", 58000),
    (15, "Oscar",  "IT",    "Delhi",     "2024-01-15", 95000),
    (16, "Paul",   "Sales", "Delhi",     "2024-01-16", 72000),
    (17, "Quinn",  "HR",    "Mumbai",    "2024-01-17", 67000),
    (18, "Rose",   "IT",    "Bangalore", "2024-01-18", 88000),
    (19, "Sam",    "Sales", "Mumbai",    "2024-01-19", 79000),
    (20, "Tina",   "HR",    "Delhi",     "2024-01-20", 61000)
]

columns = ["emp_id", "name", "department", "city", "date", "salary"]

df = spark.createDataFrame(data, columns)

df = df.withColumn("date", F.to_date("date", "yyyy-MM-dd"))

df.show(truncate=False)
df.printSchema()

+------+-------+----------+---------+----------+------+
|emp_id|name   |department|city     |date      |salary|
+------+-------+----------+---------+----------+------+
|1     |Alice  |HR        |Mumbai   |2024-01-01|50000 |
|2     |Bob    |IT        |Bangalore|2024-01-02|70000 |
|3     |Charlie|IT        |Mumbai   |2024-01-03|80000 |
|4     |David  |HR        |Delhi    |2024-01-04|60000 |
|5     |Eve    |Sales     |Mumbai   |2024-01-05|75000 |
|6     |Frank  |Sales     |Delhi    |2024-01-06|72000 |
|7     |Grace  |IT        |Bangalore|2024-01-07|90000 |
|8     |Helen  |HR        |Mumbai   |2024-01-08|65000 |
|9     |Ivy    |IT        |Delhi    |2024-01-09|70000 |
|10    |Jack   |Sales     |Bangalore|2024-01-10|68000 |
|11    |Kevin  |HR        |Delhi    |2024-01-11|60000 |
|12    |Lara   |IT        |Mumbai   |2024-01-12|85000 |
|13    |Mona   |Sales     |Mumbai   |2024-01-13|75000 |
|14    |Nina   |HR        |Bangalore|2024-01-14|58000 |
|15    |Oscar  |IT        |Delhi    |2024-01-15|

from pyspark.sql.window import Window 
window_spac_sal_desc=Window.partitionBy("department").orderBy(F.col("salary").desc())

In [12]:
# Ranking per department 
from pyspark.sql.window import Window 
window_spac_sal_desc=Window.partitionBy("department").orderBy(F.col("salary").desc())


# Ranking per deparment + city 

window_dept_city=Window.partitionBy("department","city").orderBy(F.col("salary").desc())

# Top emp per deparment 

df.withColumn(
    "rank",
    F.row_number().over(window_spac_sal_desc)
).filter("rank=1").show()


In [13]:
df.withColumn(
    "rank",
    F.row_number().over(window_spac_sal_desc)
).filter("rank=1").show()

+------+-----+----------+------+----------+------+----+
|emp_id| name|department|  city|      date|salary|rank|
+------+-----+----------+------+----------+------+----+
|    17|Quinn|        HR|Mumbai|2024-01-17| 67000|   1|
|    15|Oscar|        IT| Delhi|2024-01-15| 95000|   1|
|    19|  Sam|     Sales|Mumbai|2024-01-19| 79000|   1|
+------+-----+----------+------+----------+------+----+



# Total salary per department



# Find top 2 cities with highest total salary per deparment 

agg_df=df.groupBy("department","city").agg(
    F.sum("salary").alias("total_salary")
)

window_spaec=Window.partitionBy("department").orderBy(F.col("total_salary").desc())

ranked_df=agg_df.withColumn(
    "city_rnk",
    F.row_number().over(window_spaec)
)

result_df=ranked_df.filter(
    F.col("city_rnk")<=2
)

result_df.show()

In [16]:
agg_df=df.groupBy("department","city").agg(
    F.sum("salary").alias("total_salary")
)

window_spaec=Window.partitionBy("department").orderBy(F.col("total_salary").desc())

ranked_df=agg_df.withColumn(
    "city_rnk",
    F.row_number().over(window_spaec)
)

result_df=ranked_df.filter(
    F.col("city_rnk")<=2
)

result_df.show()

+----------+---------+------------+--------+
|department|     city|total_salary|city_rnk|
+----------+---------+------------+--------+
|        HR|   Mumbai|      182000|       1|
|        HR|    Delhi|      181000|       2|
|        IT|Bangalore|      248000|       1|
|        IT|   Mumbai|      165000|       2|
|     Sales|   Mumbai|      229000|       1|
|     Sales|    Delhi|      144000|       2|
+----------+---------+------------+--------+



# Find the emp with same salary but diffrent row_number 

In [20]:
df_dup=df.union(df.filter("emp_id IN (1,2,3)"))
df_dup.show(25,truncate=False)

+------+-------+----------+---------+----------+------+
|emp_id|name   |department|city     |date      |salary|
+------+-------+----------+---------+----------+------+
|1     |Alice  |HR        |Mumbai   |2024-01-01|50000 |
|2     |Bob    |IT        |Bangalore|2024-01-02|70000 |
|3     |Charlie|IT        |Mumbai   |2024-01-03|80000 |
|4     |David  |HR        |Delhi    |2024-01-04|60000 |
|5     |Eve    |Sales     |Mumbai   |2024-01-05|75000 |
|6     |Frank  |Sales     |Delhi    |2024-01-06|72000 |
|7     |Grace  |IT        |Bangalore|2024-01-07|90000 |
|8     |Helen  |HR        |Mumbai   |2024-01-08|65000 |
|9     |Ivy    |IT        |Delhi    |2024-01-09|70000 |
|10    |Jack   |Sales     |Bangalore|2024-01-10|68000 |
|11    |Kevin  |HR        |Delhi    |2024-01-11|60000 |
|12    |Lara   |IT        |Mumbai   |2024-01-12|85000 |
|13    |Mona   |Sales     |Mumbai   |2024-01-13|75000 |
|14    |Nina   |HR        |Bangalore|2024-01-14|58000 |
|15    |Oscar  |IT        |Delhi    |2024-01-15|

window_spec=Window.partitionBy("department").orderBy(F.col("salary").desc())

df_ranked=df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
)



In [19]:
df_dup.count()

23

In [21]:
window_spec=Window.partitionBy("department").orderBy(F.col("salary").desc())

df_ranked=df_dup.withColumn(
    "row_num",
    F.row_number().over(window_spec)
)
df_ranked.show(25,truncate=False)

+------+-------+----------+---------+----------+------+-------+
|emp_id|name   |department|city     |date      |salary|row_num|
+------+-------+----------+---------+----------+------+-------+
|17    |Quinn  |HR        |Mumbai   |2024-01-17|67000 |1      |
|8     |Helen  |HR        |Mumbai   |2024-01-08|65000 |2      |
|20    |Tina   |HR        |Delhi    |2024-01-20|61000 |3      |
|4     |David  |HR        |Delhi    |2024-01-04|60000 |4      |
|11    |Kevin  |HR        |Delhi    |2024-01-11|60000 |5      |
|14    |Nina   |HR        |Bangalore|2024-01-14|58000 |6      |
|1     |Alice  |HR        |Mumbai   |2024-01-01|50000 |7      |
|1     |Alice  |HR        |Mumbai   |2024-01-01|50000 |8      |
|15    |Oscar  |IT        |Delhi    |2024-01-15|95000 |1      |
|7     |Grace  |IT        |Bangalore|2024-01-07|90000 |2      |
|18    |Rose   |IT        |Bangalore|2024-01-18|88000 |3      |
|12    |Lara   |IT        |Mumbai   |2024-01-12|85000 |4      |
|3     |Charlie|IT        |Mumbai   |202

window_salary=window.partitionBy("department","salary")
df_final=df_ranked.withColumn(
    "salary_count",
    F.count("*").over(window_salary)
)

result=df_final.filter(
    F.col("salary_count")>1
)
result.show()

In [22]:
window_salary=Window.partitionBy("department","salary")
df_final=df_ranked.withColumn(
    "salary_count",
    F.count("*").over(window_salary)
)

result=df_final.filter(
    F.col("salary_count")>1
)
result.show()

+------+-------+----------+---------+----------+------+-------+------------+
|emp_id|   name|department|     city|      date|salary|row_num|salary_count|
+------+-------+----------+---------+----------+------+-------+------------+
|     1|  Alice|        HR|   Mumbai|2024-01-01| 50000|      7|           2|
|     1|  Alice|        HR|   Mumbai|2024-01-01| 50000|      8|           2|
|     4|  David|        HR|    Delhi|2024-01-04| 60000|      4|           2|
|    11|  Kevin|        HR|    Delhi|2024-01-11| 60000|      5|           2|
|     2|    Bob|        IT|Bangalore|2024-01-02| 70000|      7|           3|
|     9|    Ivy|        IT|    Delhi|2024-01-09| 70000|      8|           3|
|     2|    Bob|        IT|Bangalore|2024-01-02| 70000|      9|           3|
|     3|Charlie|        IT|   Mumbai|2024-01-03| 80000|      5|           2|
|     3|Charlie|        IT|   Mumbai|2024-01-03| 80000|      6|           2|
|     6|  Frank|     Sales|    Delhi|2024-01-06| 72000|      4|           2|